In [45]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score

In [46]:
matches = pd.read_csv("matches.csv", index_col=0)

In [47]:
#Convert non-numerical data to numerical data to be used by the model
matches["Date"] = pd.to_datetime(matches["Date"])
matches["Home_or_Away_numeric"] = matches["Home_or_Away"].astype("category").cat.codes
matches["Team_numeric"] = matches["Team"].astype("category").cat.codes
matches["Opponent_numeric"] = matches["Opponent"].astype("category").cat.codes
matches["Target"] = (matches["Win_Loss"] == "W").astype(int)
matches

,Date,Home_or_Away,Team,Opponent,Win_Loss,Rest_Days,Points,Points_Allowed,Point_Differential,FG_Made,...,Rebounds,Assists,Steals,Blocks,Turnovers,Season,Home_or_Away_numeric,Team_numeric,Opponent_numeric,Target
Id,,,,,,,,,,,,,,,,,,,,,
1,2023-10-24,Home,GSW,PHX,L,NaN,104,108,-4,36,...,49,19,11,6,11,2023-24,1,9,23,0
2,2023-10-24,Away,LAL,DEN,L,NaN,107,119,-12,41,...,44,23,5,4,12,2023-24,0,13,7,0
3,2023-10-24,Away,PHX,GSW,W,NaN,108,104,4,42,...,60,23,5,7,19,2023-24,0,23,9,1
4,2023-10-24,Home,DEN,LAL,W,NaN,119,107,12,48,...,42,29,9,6,12,2023-24,1,7,13,1
5,2023-10-25,Away,BOS,NYK,W,NaN,108,104,4,37,...,46,18,6,11,13,2023-24,0,2,19,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7376,2026-04-12,Home,POR,SAC,W,2.0,122,110,12,47,...,46,28,9,3,12,2025-26,1,24,25,1
7377,2026-04-12,Away,SAC,POR,L,2.0,110,122,-12,40,...,47,24,7,8,17,2025-26,0,25,24,0
7378,2026-04-12,Home,SAS,DEN,L,2.0,118,128,-10,45,...,45,27,5,5,9,2025-26,1,26,7,0


In [48]:
# Train Random Forest model on the first 2 season of the data set (2023-24, 2024-25)
# Returns combined dataframe and precision and accuracy metrics
rf = RandomForestClassifier(n_estimators=50, min_samples_split=10, random_state=1)

def make_predictions(data, predictors):
    train = data[data["Date"] < "2025-10-23"]
    test = data[data["Date"] >= "2025-10-23"]
    rf.fit(train[predictors], train["Target"])
    predictions = rf.predict(test[predictors])

    combined = pd.DataFrame(dict(actual=test["Target"], prediction=predictions), index=test.index) 
    precision = precision_score(test["Target"], predictions)
    accuracy = accuracy_score(test["Target"], predictions) 

    return combined, precision, accuracy

In [49]:
# Function to calculate rolling averages 
def rolling_averages(group, cols, new_cols):
    group = group.sort_values("Date")
    rolling_stats = group[cols].rolling(3 , closed="left").mean() 
    group[new_cols] = rolling_stats 
    group = group.dropna(subset=new_cols) 
    return group

In [50]:
#Compute rolling averages
cols = ["FG_Made","FG_Attempted","3PT_FG_Made","3PT_FG_Attempted", "Point_Differential", "Target"]
new_cols = [f"{c}_rolling" for c in cols]
matches_rolling = matches.groupby("Team").apply(lambda x: rolling_averages(x, cols, new_cols))
matches_rolling = matches_rolling.reset_index(level='Team')  
matches_rolling.index = range(matches_rolling.shape[0])
matches_rolling

,Team,Date,Home_or_Away,Opponent,Win_Loss,Rest_Days,Points,Points_Allowed,Point_Differential,FG_Made,...,Home_or_Away_numeric,Team_numeric,Opponent_numeric,Target,FG_Made_rolling,FG_Attempted_rolling,3PT_FG_Made_rolling,3PT_FG_Attempted_rolling,Point_Differential_rolling,Target_rolling
0,ATL,2023-10-30,Home,MIN,W,1.0,127,113,14,48,...,1,0,17,1,42.666667,91.000000,10.666667,32.666667,1.666667,0.333333
1,ATL,2023-11-01,Home,WAS,W,2.0,130,121,9,46,...,1,0,29,1,45.666667,88.666667,13.666667,33.000000,8.333333,0.666667
2,ATL,2023-11-04,Away,NOP,W,3.0,123,105,18,45,...,0,0,18,1,47.000000,90.333333,12.666667,33.000000,13.333333,1.000000
3,ATL,2023-11-06,Away,OKC,L,2.0,117,126,-9,38,...,0,0,20,0,46.333333,90.333333,12.333333,34.333333,13.666667,1.000000
4,ATL,2023-11-09,Away,ORL,W,3.0,120,119,1,41,...,0,0,21,1,43.000000,95.666667,12.333333,38.333333,6.000000,0.666667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7285,WAS,2026-04-05,Away,BKN,L,1.0,115,121,-6,44,...,0,29,1,0,47.000000,98.333333,13.333333,38.333333,-19.000000,0.000000
7286,WAS,2026-04-07,Home,CHI,L,2.0,98,129,-31,37,...,1,29,4,0,48.000000,93.666667,14.000000,35.000000,-14.666667,0.000000
7287,WAS,2026-04-09,Home,CHI,L,2.0,108,119,-11,38,...,1,29,4,0,43.666667,90.333333,12.333333,34.333333,-17.666667,0.000000
7288,WAS,2026-04-10,Home,MIA,L,1.0,117,140,-23,46,...,1,29,15,0,39.666667,87.000000,10.000000,33.666667,-16.000000,0.000000


In [44]:
predictors = ["Home_or_Away_numeric", "Opponent_numeric", "Rest_Days"]
combined, precision, accuracy = make_predictions(matches, predictors)
rolling_combined, rolling_precision, rolling_accuracy = make_predictions(matches_rolling, predictors + new_cols)
precision, accuracy, rolling_precision, rolling_accuracy

(0.5502606105733433, 0.5555098684210527, 0.5898305084745763, 0.587171052631579)